# Ejemplo de uso del modulo con dataset del sistema ADATRAP

## Nivel 1

### Configuración inicial

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "ADATRAP" 

In [2]:
import pandas as pd
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

### Carga de datos

In [19]:
import unicodedata
import yaml

from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.transforms.spatial import project_xy_to_latlon



# ------------------------------------------------------------
# 1) Carga de archivos
# ------------------------------------------------------------
adatrap_viajes = pd.read_csv(
    DATA_PATH / "semana_2019_05_13.viajes",
    sep="|",
    dtype=str,
    low_memory=False,
)
adatrap_viajes = adatrap_viajes.sample(n=5000).sort_index()

adatrap_stops = pd.read_csv(
    DATA_PATH / "DIC_777_fixed.csv",
    sep=",",
    dtype=str,
    low_memory=False,
)

with open(DATA_PATH / "stage_layout.yaml", "r", encoding="utf-8") as f:
    stage_layout = yaml.safe_load(f)

with open(DATA_PATH / "domains.yaml", "r", encoding="utf-8") as f:
    adatrap_domains = yaml.safe_load(f)

### Pre-procesamiento de datos

In [27]:
# ------------------------------------------------------------
# 2) Funciones pequeñas y necesarias para estaciones
# ------------------------------------------------------------
def fix_station(station: str) -> str:
    if pd.isna(station):
        return pd.NA

    station = str(station).strip()
    parts = station.split("-")

    if len(parts) < 2:
        return station

    if not parts[-1].isdigit():
        parts[-1], parts[-2] = parts[-2], parts[-1]
        return "-".join(parts)

    return station


def standardize_spanish_station(text: str) -> str:
    if pd.isna(text):
        return pd.NA

    text = str(text).strip()
    parts = text.split("-")

    if len(parts) < 2:
        normalized = unicodedata.normalize("NFKD", text)
        standardized = "".join(
            c for c in normalized if not unicodedata.combining(c)
        )
        standardized = standardized.replace("`", "").upper().strip()
        return standardized

    return text


# ------------------------------------------------------------
# 3) Selección simple de columnas comunes del viaje
# ------------------------------------------------------------
trip_level_cols = [c for c in stage_layout["trip_level"] if c in adatrap_viajes.columns]

# Me quedo con la mayoría de columnas comunes del viaje.
df_adatrap_trips_lvl1 = adatrap_viajes[trip_level_cols].copy()

# ------------------------------------------------------------
# 4) Construcción manual de algunos campos "genéricos" del viaje
#    usando primera subida y última bajada válidas.
# ------------------------------------------------------------
first_stage_origin_cols = [
    "paraderosubida_1era",
    "tiemposubida_1era",
    "zona777subida_1era",
    "linea_metro_subida_1",
]

last_stage_destination_cols_candidates = [
    ("paraderobajada_4ta", "tiempobajada_4ta", "zona777bajada_4ta", "linea_metro_bajada_4"),
    ("paraderobajada_3era", "tiempobajada_3era", "zona777bajada_3era", "linea_metro_bajada_3"),
    ("paraderobajada_2da", "tiempobajada_2da", "zona777bajada_2da", "linea_metro_bajada_2"),
    ("paraderobajada_1era", "tiempobajada_1era", "zona777bajada_1era", "linea_metro_bajada_1"),
]

# Subida del viaje: tomo la primera etapa
df_adatrap_trips_lvl1["paraderosubida_viaje"] = df_adatrap_trips_lvl1.index.map(
    lambda i: adatrap_viajes.loc[i, "paraderosubida_1era"]
)
df_adatrap_trips_lvl1["tiemposubida_viaje"] = df_adatrap_trips_lvl1.index.map(
    lambda i: adatrap_viajes.loc[i, "tiemposubida_1era"]
)
df_adatrap_trips_lvl1["zona777subida_viaje"] = df_adatrap_trips_lvl1.index.map(
    lambda i: adatrap_viajes.loc[i, "zona777subida_1era"]
)
df_adatrap_trips_lvl1["linea_metro_subida_viaje"] = df_adatrap_trips_lvl1.index.map(
    lambda i: adatrap_viajes.loc[i, "linea_metro_subida_1"]
)

# Bajada del viaje: tomo la última bajada válida entre etapa 4 -> 1
paraderobajada_viaje = []
tiempobajada_viaje = []
zona777bajada_viaje = []
linea_metro_bajada_viaje = []

for _, row in adatrap_viajes.iterrows():
    chosen = (pd.NA, pd.NA, pd.NA, pd.NA)

    for stop_col, time_col, zona_col, metro_col in last_stage_destination_cols_candidates:
        stop_val = row.get(stop_col)
        if pd.notna(stop_val) and str(stop_val).strip() != "-":
            chosen = (
                row.get(stop_col),
                row.get(time_col),
                row.get(zona_col),
                row.get(metro_col),
            )
            break

    paraderobajada_viaje.append(chosen[0])
    tiempobajada_viaje.append(chosen[1])
    zona777bajada_viaje.append(chosen[2])
    linea_metro_bajada_viaje.append(chosen[3])

df_adatrap_trips_lvl1["paraderobajada_viaje"] = paraderobajada_viaje
df_adatrap_trips_lvl1["tiempobajada_viaje"] = tiempobajada_viaje
df_adatrap_trips_lvl1["zona777bajada_viaje"] = zona777bajada_viaje
df_adatrap_trips_lvl1["linea_metro_bajada_viaje"] = linea_metro_bajada_viaje

# ------------------------------------------------------------
# 5) Limpieza mínima de nombres de estaciones/paraderos
# ------------------------------------------------------------
df_adatrap_trips_lvl1["paraderosubida_viaje"] = (
    df_adatrap_trips_lvl1["paraderosubida_viaje"].map(fix_station).map(standardize_spanish_station)
)
df_adatrap_trips_lvl1["paraderobajada_viaje"] = (
    df_adatrap_trips_lvl1["paraderobajada_viaje"].map(fix_station).map(standardize_spanish_station)
)

# ------------------------------------------------------------
# 6) Preparación de tabla de paraderos/estaciones y merge de coords
# ------------------------------------------------------------
stops_table = adatrap_stops.rename(columns={"parada/est.metro": "parada"}).copy()
stops_table["parada"] = stops_table["parada"].map(standardize_spanish_station)

# Coordenadas de subida
df_adatrap_trips_lvl1 = df_adatrap_trips_lvl1.merge(
    stops_table[["parada", "x", "y"]].rename(columns={"x": "subida_x", "y": "subida_y"}),
    how="left",
    left_on="paraderosubida_viaje",
    right_on="parada",
)
df_adatrap_trips_lvl1 = df_adatrap_trips_lvl1.drop(columns=["parada"])

# Coordenadas de bajada
df_adatrap_trips_lvl1 = df_adatrap_trips_lvl1.merge(
    stops_table[["parada", "x", "y"]].rename(columns={"x": "bajada_x", "y": "bajada_y"}),
    how="left",
    left_on="paraderobajada_viaje",
    right_on="parada",
)
df_adatrap_trips_lvl1 = df_adatrap_trips_lvl1.drop(columns=["parada"])

# ------------------------------------------------------------
# 7) Conversión XY -> lon/lat usando el helper del módulo
# ------------------------------------------------------------
df_adatrap_trips_lvl1 = project_xy_to_latlon(
    df_adatrap_trips_lvl1,
    x_col="subida_x",
    y_col="subida_y",
    source_crs="EPSG:5361",
    lon_col="origin_longitude",
    lat_col="origin_latitude",
    keep_debug_cols=False,
    drop_input_cols=False,
)

df_adatrap_trips_lvl1 = project_xy_to_latlon(
    df_adatrap_trips_lvl1,
    x_col="bajada_x",
    y_col="bajada_y",
    source_crs="EPSG:5361",
    lon_col="destination_longitude",
    lat_col="destination_latitude",
    keep_debug_cols=False,
    drop_input_cols=False,
)

# ------------------------------------------------------------
# 8) Campos mínimos para importar
#    Por ahora no intento conservar todo.
# ------------------------------------------------------------
df_adatrap_trips_lvl1["movement_id_src"] = df_adatrap_trips_lvl1.index.map(lambda i: f"m{i}")

cols_for_import_lvl1 = [
    "movement_id_src",
    "id",
    "tipodia",
    "proposito",
    "periodomediodeviaje",
    "factorexpansion",
    "ultimaetapaconbajada",
    "tipo_corte_etapa_viaje",
    "paraderosubida_viaje",
    "paraderobajada_viaje",
    "linea_metro_subida_viaje",
    "linea_metro_bajada_viaje",
    "origin_longitude",
    "origin_latitude",
    "destination_longitude",
    "destination_latitude",
    "tiemposubida_viaje",
    "tiempobajada_viaje",
]

df_adatrap_trips_lvl1 = df_adatrap_trips_lvl1[cols_for_import_lvl1].copy()

display(df_adatrap_trips_lvl1.head())

,movement_id_src,id,tipodia,proposito,periodomediodeviaje,factorexpansion,ultimaetapaconbajada,tipo_corte_etapa_viaje,paraderosubida_viaje,paraderobajada_viaje,linea_metro_subida_viaje,linea_metro_bajada_viaje,origin_longitude,origin_latitude,destination_longitude,destination_latitude,tiemposubida_viaje,tiempobajada_viaje
0,m0,19949778,LABORAL,OTROS,09 - PUNTA TARDE,1.3320,1,MS_M_B,SAN MIGUEL,CAL Y CANTO,L2,L2,-70.650829,-33.488793,-70.653222,-33.432893,2019-05-13 19:30:12,2019-05-13 19:43:36
1,m1,19949778,LABORAL,OTROS,09 - PUNTA TARDE,1.3320,1,MS_M_B,SAN MIGUEL,CAL Y CANTO,L2,L2,-70.650829,-33.488793,-70.653222,-33.432893,2019-05-13 19:30:12,2019-05-13 19:43:36
2,m2,19949778,LABORAL,OTROS,09 - PUNTA TARDE,1.3320,1,MS_M_B,SAN MIGUEL,CAL Y CANTO,L2,L2,-70.650829,-33.488793,-70.653222,-33.432893,2019-05-13 19:30:12,2019-05-13 19:43:36
3,m3,19949778,LABORAL,OTROS,09 - PUNTA TARDE,1.3320,1,MS_M_B,SAN MIGUEL,CAL Y CANTO,L2,L2,-70.650829,-33.488793,-70.653222,-33.432893,2019-05-13 19:30:12,2019-05-13 19:43:36
4,m4,19959586,LABORAL,HOGAR,08 - FUERA DE PUNTA TARDE,1.3883,1,UE,E-17-140-OP-60,L-17-40-SN-5,-,-,-70.581689,-33.412976,-70.583317,-33.413207,2019-05-13 16:50:44,2019-05-13 16:53:29


### Objetos para el Import trips

In [28]:
ADATRAP_TRIPS_LVL1_SCHEMA = TripSchema(
    version="adatrap-trips-lvl1-simplificado",
    fields={
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "trip_id": FieldSpec("trip_id", "string", required=True),
        "movement_seq": FieldSpec("movement_seq", "int", required=True),

        "user_id": FieldSpec("user_id", "string", required=True),

        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),

        "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
        "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),

        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["home", "work", "no_destination", "other"],
                extendable=False,
            ),
        ),
        "day_type": FieldSpec(
            "day_type",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["weekday", "weekend"],
                extendable=False,
            ),
        ),
        "time_period": FieldSpec(
            "time_period",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["evening", "night", "morning", "midday", "afternoon"],
                extendable=False,
            ),
        ),

        "trip_weight": FieldSpec("trip_weight", "float", required=False),
        "ultimaetapaconbajada": FieldSpec(
            "ultimaetapaconbajada",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "tipo_corte_etapa_viaje": FieldSpec(
            "tipo_corte_etapa_viaje",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
    },
    required=[
        "movement_id",
        "trip_id",
        "movement_seq",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ],
)

ADATRAP_TRIPS_LVL1_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone="America/Santiago",
)

ADATRAP_TRIPS_LVL1_FIELD_CORRESPONDENCE = {
    "movement_id": "movement_id_src",
    "user_id": "id",
    "origin_time_utc": "tiemposubida_viaje",
    "destination_time_utc": "tiempobajada_viaje",
    "purpose": "proposito",
    "day_type": "tipodia",
    "time_period": "periodomediodeviaje",
    "trip_weight": "factorexpansion",
}
test = ["evening", "night", "morning", "midday", "afternoon"]
ADATRAP_TRIPS_LVL1_VALUE_CORRESPONDENCE = {
    "purpose": {
        "HOGAR": "home",
        "TRABAJO": "work",
        "OTROS": "other",
        "SINBAJADA": "no_destination",
        "MENOS1MINUTO" : "other"
    },
    "day_type": {
        "LABORAL": "weekday",
        "SABADO": "weekend",
        "DOMINGO": "weekend",
    },
    "time_period": {
        "01 - PRE NOCTURNO": "evening",
        "01 - PRE NOCTURNO DOMINGO": "evening",
        "01 - PRE NOCTURNO SABADO": "evening",
        "02 - NOCTURNO": "night",
        "02 - NOCTURNO DOMINGO": "night",
        "02 - NOCTURNO SABADO": "night",
        "03 - TRANSICION DOMINGO MANANA": "morning",
        "03 - TRANSICION SABADO MANANA": "morning",
        "04 - MANANA DOMINGO": "morning",
        "04 - PUNTA MANANA": "morning",
        "04 - PUNTA MANANA SABADO": "morning",
        "05 - MANANA SABADO": "morning",
        "05 - TRANSICION PUNTA MANANA": "morning",
        "06 - FUERA DE PUNTA MANANA": "morning",
        "05 - MEDIODIA DOMINGO": "midday",
        "06 - PUNTA MEDIODIA SABADO": "midday",
        "06 - TARDE DOMINGO": "midday",
        "07 - PUNTA MEDIODIA": "midday",
        "06 - TARDE DOMINGO": "afternoon",
        "07 - TARDE SABADO": "afternoon",
        "08 - FUERA DE PUNTA TARDE": "afternoon",
        "09 - PUNTA TARDE": "afternoon",
        "10 - TRANSICION PUNTA TARDE": "afternoon",
    },
}

ADATRAP_TRIPS_LVL1_PROVENANCE = {
    "source": {
        "name": "ADATRAP",
        "entity": "trips",
        "version": "perfil_semana",
    },
    "notes": [
        "nivel 1 simplificado",
        "viaje resumido desde primera subida y última bajada",
        "uso simple de stage_layout.yaml y helper espacial",
    ],
}

### Hacemos el import trips

In [29]:
trips_adatrap_lvl1, report_adatrap_lvl1 = import_trips_from_dataframe(
    df_adatrap_trips_lvl1.head(1000),
    schema=ADATRAP_TRIPS_LVL1_SCHEMA,
    source_name="ADATRAP trips - nivel 1 simplificado",
    options=ADATRAP_TRIPS_LVL1_OPTIONS,
    field_correspondence=ADATRAP_TRIPS_LVL1_FIELD_CORRESPONDENCE,
    value_correspondence=ADATRAP_TRIPS_LVL1_VALUE_CORRESPONDENCE,
    provenance=ADATRAP_TRIPS_LVL1_PROVENANCE,
    h3_resolution=8,
)

display(trips_adatrap_lvl1.data.head())
display(report_adatrap_lvl1.issues[:10])

,movement_id,trip_id,movement_seq,user_id,day_type,purpose,time_period,trip_weight,ultimaetapaconbajada,tipo_corte_etapa_viaje,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc
0,m0,m0,0,19949778,weekday,other,afternoon,1.3320,1,MS_M_B,-70.650829,-33.488793,-70.653222,-33.432893,2019-05-13 23:30:12+00:00,2019-05-13 23:43:36+00:00
1,m1,m1,0,19949778,weekday,other,afternoon,1.3320,1,MS_M_B,-70.650829,-33.488793,-70.653222,-33.432893,2019-05-13 23:30:12+00:00,2019-05-13 23:43:36+00:00
2,m2,m2,0,19949778,weekday,other,afternoon,1.3320,1,MS_M_B,-70.650829,-33.488793,-70.653222,-33.432893,2019-05-13 23:30:12+00:00,2019-05-13 23:43:36+00:00
3,m3,m3,0,19949778,weekday,other,afternoon,1.3320,1,MS_M_B,-70.650829,-33.488793,-70.653222,-33.432893,2019-05-13 23:30:12+00:00,2019-05-13 23:43:36+00:00
4,m4,m4,0,19959586,weekday,home,afternoon,1.3883,1,UE,-70.581689,-33.412976,-70.583317,-33.413207,2019-05-13 20:50:44+00:00,2019-05-13 20:53:29+00:00


[Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'ultimaetapaconbajada' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='ultimaetapaconbajada', source_field=None, row_count=None, details={'field': 'ultimaetapaconbajada', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'tipo_corte_etapa_viaje' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='tipo_corte_etapa_viaje', source_field=None, row_count=None, details={'field': 'tipo_corte_etapa_viaje', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='warning', code='DOM.POLICY.FIELD_NOT_EXTENDABLE', message="El campo 'time_period' no admite extensión de dominio (extendable=False); los valores fuera de dominio se mapearán a unknown.", field='time_period', source_field=None, row_count=Non

## Nivel 2

### Carga de datos

In [31]:
import yaml

from pylondrina.importing import ImportOptions
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.sources.profile import SourceProfile
from pylondrina.sources.helpers import import_trips_from_profile

#adatrap_viajes = pd.read_csv(DATA_PATH / "semana_2019_05_13.viajes", sep="|", dtype=str, low_memory=False)
#adatrap_stops = pd.read_csv(DATA_PATH / "DIC_777_fixed.csv", sep=",", dtype=str, low_memory=False)

with open(DATA_PATH / "stage_layout.yaml", "r", encoding="utf-8") as f:
    stage_layout = yaml.safe_load(f)

with open(DATA_PATH / "domains.yaml", "r", encoding="utf-8") as f:
    adatrap_domains = yaml.safe_load(f)

trip_level_cols = [c for c in stage_layout["trip_level"] if c in adatrap_viajes.columns]
stage_1_cols = [c for c in stage_layout["stage_1"] if c in adatrap_viajes.columns]
stage_2_cols = [c for c in stage_layout["stage_2"] if c in adatrap_viajes.columns]
stage_3_cols = [c for c in stage_layout["stage_3"] if c in adatrap_viajes.columns]
stage_4_cols = [c for c in stage_layout["stage_4"] if c in adatrap_viajes.columns]

### Schema, options y correspondencias

In [32]:
ADATRAP_STAGES_LVL2_SCHEMA = TripSchema(
    version="adatrap-stages-lvl2-sourceprofile",
    fields={
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "movement_seq": FieldSpec("movement_seq", "int", required=True),
        "user_id": FieldSpec("user_id", "string", required=True),

        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),

        "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
        "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),

        "origin_stop_code": FieldSpec(
            "origin_stop_code",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "destination_stop_code": FieldSpec(
            "destination_stop_code",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["bus", "metro", "train", "other"],
                extendable=False,
            ),
        ),
        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["home", "work", "other"],
                extendable=False,
            ),
        ),
        "day_type": FieldSpec(
            "day_type",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["weekday", "weekend"],
                extendable=False,
            ),
        ),
        "time_period": FieldSpec(
            "time_period",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "trip_weight": FieldSpec("trip_weight", "float", required=False),
        "origin_municipality": FieldSpec(
            "origin_municipality",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
        "destination_municipality": FieldSpec(
            "destination_municipality",
            "categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        ),
    },
    required=[
        "movement_id",
        "movement_seq",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ],
)

ADATRAP_STAGES_LVL2_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=False,
    source_timezone="America/Santiago",
)

ADATRAP_STAGES_LVL2_FIELD_CORRESPONDENCE = {
    "movement_id": "movement_id_src",
    "movement_seq": "numero_etapa",
    "user_id": "id",
    "origin_time_utc": "tiemposubida_etapa",
    "destination_time_utc": "tiempobajada_etapa",
    "origin_stop_code": "paraderosubida_etapa",
    "destination_stop_code": "paraderobajada_etapa",
    "mode": "tipotransporte_etapa",
    "purpose": "proposito",
    "day_type": "tipodia",
    "time_period": "periodomediodeviaje",
    "trip_weight": "factorexpansion",
    "origin_municipality": "comunasubida",
    "destination_municipality": "comunabajada",
}

ADATRAP_STAGES_LVL2_VALUE_CORRESPONDENCE = {
    "mode": {
        "BUS": "bus",
        "METRO": "metro",
        "METROTREN": "train",
        "ZP": "other",
    },
    "purpose": {
        "HOGAR": "home",
        "TRABAJO": "work",
        "OTROS": "other",
        "SINBAJADA": "other",
    },
    "day_type": {
        "LABORAL": "weekday",
        "SABADO": "weekend",
        "DOMINGO": "weekend",
    },
    "time_period": {
        "PRE NOCTURNO": "evening",
        "NOCTURNO": "night",
        "MANANA": "morning",
        "MEDIODIA": "midday",
        "TARDE": "afternoon",
    },
}

ADATRAP_STAGES_LVL2_PROVENANCE = {
    "source": {
        "name": "ADATRAP",
        "entity": "stages",
        "version": "perfil_semana",
    },
    "notes": [
        "nivel 2 con SourceProfile",
        "wide->long simple guiado por stage_layout.yaml",
    ],
}

### Uso de SourceProfiles y preprocess

In [33]:
def fix_station(station: str) -> str:
    if pd.isna(station):
        return pd.NA

    station = str(station).strip()
    parts = station.split("-")

    if len(parts) < 2:
        return station

    if not parts[-1].isdigit():
        parts[-1], parts[-2] = parts[-2], parts[-1]
        return "-".join(parts)

    return station


def standardize_spanish_station(text: str) -> str:
    if pd.isna(text):
        return pd.NA

    text = str(text).strip()
    parts = text.split("-")

    if len(parts) < 2:
        normalized = unicodedata.normalize("NFKD", text)
        standardized = "".join(c for c in normalized if not unicodedata.combining(c))
        return standardized.replace("`", "").upper().strip()

    return text


def preprocess_adatrap_stages_lvl2(df_wide: pd.DataFrame) -> pd.DataFrame:
    rows = []

    # Tabla de estaciones/paraderos normalizada
    stops_table = adatrap_stops.rename(columns={"parada/est.metro": "parada"}).copy()
    stops_table["parada"] = stops_table["parada"].map(standardize_spanish_station)

    stage_groups = [
        (1, stage_1_cols),
        (2, stage_2_cols),
        (3, stage_3_cols),
        (4, stage_4_cols),
    ]

    for _, row in df_wide.iterrows():
        for stage_idx, stage_cols in stage_groups:
            stage_data = {c: row.get(c) for c in stage_cols}

            # Si esta etapa no tiene casi nada, la salto.
            if all(pd.isna(v) or str(v).strip() == "-" for v in stage_data.values()):
                continue

            out = {c: row.get(c) for c in trip_level_cols}
            out["numero_etapa"] = stage_idx

            # Renombro columnas numeradas a nombres genéricos de etapa.
            for c, v in stage_data.items():
                c_generic = (
                    c.replace("_1era", "_etapa")
                    .replace("_2da", "_etapa")
                    .replace("_3era", "_etapa")
                    .replace("_4ta", "_etapa")
                )
                out[c_generic] = v

            rows.append(out)

    df_long = pd.DataFrame(rows)

    # Limpieza mínima de paraderos/estaciones
    if "paraderosubida_etapa" in df_long.columns:
        df_long["paraderosubida_etapa"] = (
            df_long["paraderosubida_etapa"].map(fix_station).map(standardize_spanish_station)
        )
    if "paraderobajada_etapa" in df_long.columns:
        df_long["paraderobajada_etapa"] = (
            df_long["paraderobajada_etapa"].map(fix_station).map(standardize_spanish_station)
        )

    # Merge de coordenadas de subida
    df_long = df_long.merge(
        stops_table[["parada", "x", "y"]].rename(columns={"x": "subida_x", "y": "subida_y"}),
        how="left",
        left_on="paraderosubida_etapa",
        right_on="parada",
    )
    df_long = df_long.drop(columns=["parada"])

    # Merge de coordenadas de bajada
    df_long = df_long.merge(
        stops_table[["parada", "x", "y"]].rename(columns={"x": "bajada_x", "y": "bajada_y"}),
        how="left",
        left_on="paraderobajada_etapa",
        right_on="parada",
    )
    df_long = df_long.drop(columns=["parada"])

    # Conversión XY -> lon/lat
    df_long = project_xy_to_latlon(
        df_long,
        x_col="subida_x",
        y_col="subida_y",
        source_crs="EPSG:5361",
        lon_col="origin_longitude",
        lat_col="origin_latitude",
        keep_debug_cols=False,
        drop_input_cols=False,
    )

    df_long = project_xy_to_latlon(
        df_long,
        x_col="bajada_x",
        y_col="bajada_y",
        source_crs="EPSG:5361",
        lon_col="destination_longitude",
        lat_col="destination_latitude",
        keep_debug_cols=False,
        drop_input_cols=False,
    )

    # movement_id explícito para no depender de derivaciones
    df_long = df_long.reset_index(drop=True)
    df_long["movement_id_src"] = df_long.index.map(lambda i: f"m{i}")

    cols_for_profile = [
        "movement_id_src",
        "numero_etapa",
        "id",
        "tipodia",
        "proposito",
        "periodomediodeviaje",
        "factorexpansion",
        "comunasubida",
        "comunabajada",
        "paraderosubida_etapa",
        "paraderobajada_etapa",
        "tiemposubida_etapa",
        "tiempobajada_etapa",
        "tipotransporte_etapa",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ]

    return df_long[cols_for_profile].copy()


ADATRAP_STAGES_LVL2_PROFILE = SourceProfile(
    name="ADATRAP_STAGES_LVL2",
    description="Adaptación simplificada de ADATRAP stages con SourceProfile",
    default_field_correspondence=ADATRAP_STAGES_LVL2_FIELD_CORRESPONDENCE,
    default_value_correspondence=ADATRAP_STAGES_LVL2_VALUE_CORRESPONDENCE,
    default_options=ADATRAP_STAGES_LVL2_OPTIONS,
    preprocess=preprocess_adatrap_stages_lvl2,
    schema_override=ADATRAP_STAGES_LVL2_SCHEMA,
)

df_test = preprocess_adatrap_stages_lvl2(adatrap_viajes.head(1000))
display(df_test.head())

,movement_id_src,numero_etapa,id,tipodia,proposito,periodomediodeviaje,factorexpansion,comunasubida,comunabajada,paraderosubida_etapa,paraderobajada_etapa,tiemposubida_etapa,tiempobajada_etapa,tipotransporte_etapa,origin_longitude,origin_latitude,destination_longitude,destination_latitude
0,m0,1,19949778,LABORAL,OTROS,09 - PUNTA TARDE,1.3320,SAN MIGUEL,SANTIAGO,SAN MIGUEL,CAL Y CANTO,2019-05-13 19:30:12,2019-05-13 19:43:36,METRO,-70.650829,-33.488793,-70.653222,-33.432893
1,m1,1,19949778,LABORAL,OTROS,09 - PUNTA TARDE,1.3320,SAN MIGUEL,SANTIAGO,SAN MIGUEL,CAL Y CANTO,2019-05-13 19:30:12,2019-05-13 19:43:36,METRO,-70.650829,-33.488793,-70.653222,-33.432893
2,m2,1,19949778,LABORAL,OTROS,09 - PUNTA TARDE,1.3320,SAN MIGUEL,SANTIAGO,SAN MIGUEL,CAL Y CANTO,2019-05-13 19:30:12,2019-05-13 19:43:36,METRO,-70.650829,-33.488793,-70.653222,-33.432893
3,m3,1,19949778,LABORAL,OTROS,09 - PUNTA TARDE,1.3320,SAN MIGUEL,SANTIAGO,SAN MIGUEL,CAL Y CANTO,2019-05-13 19:30:12,2019-05-13 19:43:36,METRO,-70.650829,-33.488793,-70.653222,-33.432893
4,m4,1,19959586,LABORAL,HOGAR,08 - FUERA DE PUNTA TARDE,1.3883,LAS CONDES,LAS CONDES,E-17-140-OP-60,L-17-40-SN-5,2019-05-13 16:50:44,2019-05-13 16:53:29,BUS,-70.581689,-33.412976,-70.583317,-33.413207


### Hacemos el import

In [34]:
# inspección clara del perfil
ADATRAP_STAGES_LVL2_PROFILE.schema_override
ADATRAP_STAGES_LVL2_PROFILE.default_options
ADATRAP_STAGES_LVL2_PROFILE.default_field_correspondence
ADATRAP_STAGES_LVL2_PROFILE.default_value_correspondence

stages_adatrap_lvl2, report_stages_adatrap_lvl2 = import_trips_from_profile(
    profile=ADATRAP_STAGES_LVL2_PROFILE,
    df=adatrap_viajes.head(1000),
    source_name="ADATRAP stages - nivel 2 SourceProfile",
    provenance=ADATRAP_STAGES_LVL2_PROVENANCE,
    h3_resolution=8,
)

display(stages_adatrap_lvl2.data)
display(report_stages_adatrap_lvl2.issues[:10])

,movement_id,movement_seq,user_id,day_type,purpose,time_period,trip_weight,origin_municipality,destination_municipality,origin_stop_code,destination_stop_code,origin_time_utc,destination_time_utc,mode,origin_longitude,origin_latitude,destination_longitude,destination_latitude
0,m0,1,19949778,weekday,other,09 - PUNTA TARDE,1.3320,SAN MIGUEL,SANTIAGO,SAN MIGUEL,CAL Y CANTO,2019-05-13 23:30:12+00:00,2019-05-13 23:43:36+00:00,metro,-70.650829,-33.488793,-70.653222,-33.432893
1,m1,1,19949778,weekday,other,09 - PUNTA TARDE,1.3320,SAN MIGUEL,SANTIAGO,SAN MIGUEL,CAL Y CANTO,2019-05-13 23:30:12+00:00,2019-05-13 23:43:36+00:00,metro,-70.650829,-33.488793,-70.653222,-33.432893
2,m2,1,19949778,weekday,other,09 - PUNTA TARDE,1.3320,SAN MIGUEL,SANTIAGO,SAN MIGUEL,CAL Y CANTO,2019-05-13 23:30:12+00:00,2019-05-13 23:43:36+00:00,metro,-70.650829,-33.488793,-70.653222,-33.432893
3,m3,1,19949778,weekday,other,09 - PUNTA TARDE,1.3320,SAN MIGUEL,SANTIAGO,SAN MIGUEL,CAL Y CANTO,2019-05-13 23:30:12+00:00,2019-05-13 23:43:36+00:00,metro,-70.650829,-33.488793,-70.653222,-33.432893
4,m4,1,19959586,weekday,home,08 - FUERA DE PUNTA TARDE,1.3883,LAS CONDES,LAS CONDES,E-17-140-OP-60,L-17-40-SN-5,2019-05-13 20:50:44+00:00,2019-05-13 20:53:29+00:00,bus,-70.581689,-33.412976,-70.583317,-33.413207
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2547,m2547,1,476038738,weekday,work,08 - FUERA DE PUNTA TARDE,1.5062,LA CISTERNA,LA CISTERNA,EL PARRON,LA CISTERNA,2019-05-14 18:41:05+00:00,2019-05-14 18:43:12+00:00,metro,-70.661431,-33.526536,-70.664078,-33.537719
2548,m2548,2,476038738,weekday,work,08 - FUERA DE PUNTA TARDE,1.5062,LA CISTERNA,LA CISTERNA,T-26-12-PO-22,T-26-12-PO-30,2019-05-14 18:50:39+00:00,2019-05-14 18:51:47+00:00,bus,-70.663242,-33.537929,-70.655172,-33.539390
2549,m2549,1,476049122,weekday,work,04 - PUNTA MANANA,1.4721,QUILICURA,NUNOA,T-6-45-PO-27,T-2-7-PO-7,2019-05-14 11:58:50+00:00,2019-05-14 12:10:55+00:00,bus,-70.720840,-33.366726,-70.689771,-33.366261
2550,m2550,2,476049122,weekday,work,04 - PUNTA MANANA,1.4721,QUILICURA,NUNOA,LOS LIBERTADORES,IRARRAZAVAL,2019-05-14 12:16:55+00:00,2019-05-14 12:35:36+00:00,metro,NaN,NaN,-70.628903,-33.452442


[Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'origin_stop_code' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='origin_stop_code', source_field=None, row_count=None, details={'field': 'origin_stop_code', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'destination_stop_code' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='destination_stop_code', source_field=None, row_count=None, details={'field': 'destination_stop_code', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'time_period' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='time_period', source_field=None, row_count=None, details={'field': 'time_period

# Nivel 3

## Para datos del sistema ADATRAP - Trips / viajes

### 1) Uso normal, rápido, sin decisiones de schema/campos/valores en la factory

In [3]:
from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_adatrap.make_adatrap_trips_profile import make_adatrap_trips_profile
from scripts.source_profiles.factories_adatrap.trips_defaults import (
    ADATRAP_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
)

adatrap_viajes = pd.read_csv(
    DATA_PATH / "semana_2019_05_13.viajes",
    sep="|",
    dtype=str,
    low_memory=False,
    na_values="-"
)
adatrap_stops = pd.read_csv(
    DATA_PATH / "DIC_777_fixed.csv",
    sep=",",
    dtype=str,
    low_memory=False,
)

profile = make_adatrap_trips_profile(
    df_stops=adatrap_stops,
    stage_layout_yaml=DATA_PATH / "stage_layout.yaml",
    domains_yaml=DATA_PATH / "domains.yaml",
)

# inspección clara
#profile.schema_override
#profile.default_options
#profile.default_field_correspondence
#profile.default_value_correspondence

In [9]:
display(profile.schema_override)

{'version': '1.1',
 'required': ['movement_id',
              'user_id',
              'origin_longitude',
              'origin_latitude',
              'destination_longitude',
              'destination_latitude',
              'origin_h3_index',
              'destination_h3_index',
              'trip_id',
              'movement_seq'],
 'semantic_rules': None,
 'fields': {'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': True,
                            'constraints': None,
                            'domain': None},
            'user_id': {'name': 'user_id', 'dtype': 'string', 'required': True, 'constraints': None, 'domain': None},
            'origin_longitude': {'name': 'origin_longitude',
                                 'dtype': 'float',
                                 'required': True,
                                 'constraints': None,
                                 'domain': None},
            'origin_latitude': {'name': 'origin_latitude',
                                'dtype': 'float',
                                'required': True,
                                'constraints': None,
                                'domain': None},
            'destination_longitude': {'name': 'destination_longitude',
                                      'dtype': 'float',
                                      'required': True,
                                      'constraints': None,
                                      'domain': None},
            'destination_latitude': {'name': 'destination_latitude',
                                     'dtype': 'float',
                                     'required': True,
                                     'constraints': None,
                                     'domain': None},
            'origin_h3_index': {'name': 'origin_h3_index',
                                'dtype': 'string',
                                'required': True,
                                'constraints': None,
                                'domain': None},
            'destination_h3_index': {'name': 'destination_h3_index',
                                     'dtype': 'string',
                                     'required': True,
                                     'constraints': None,
                                     'domain': None},
            'origin_time_utc': {'name': 'origin_time_utc',
                                'dtype': 'datetime',
                                'required': False,
                                'constraints': None,
                                'domain': None},
            'destination_time_utc': {'name': 'destination_time_utc',
                                     'dtype': 'datetime',
                                     'required': False,
                                     'constraints': None,
                                     'domain': None},
            'trip_id': {'name': 'trip_id', 'dtype': 'string', 'required': True, 'constraints': None, 'domain': None},
            'movement_seq': {'name': 'movement_seq',
                             'dtype': 'int',
                             'required': True,
                             'constraints': None,
                             'domain': None},
            'mode': {'name': 'mode',
                     'dtype': 'categorical',
                     'required': False,
                     'constraints': None,
                     'domain': {'values': ['walk',
                                           'bicycle',
                                           'scooter',
                                           'motorcycle',
                                           'car',
                                           'taxi',
                                           'ride_hailing',
                                           'bus',
                                           'metro',
                                           '

In [4]:
trips_adatrap, report_adatrap = import_trips_from_profile(
    profile=profile,
    df=adatrap_viajes.head(5000),
    source_name="ADATRAP trips level-3 factory",
    provenance=ADATRAP_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)

display(trips_adatrap.data.head())
display(report_adatrap.issues)

,movement_id,trip_id,movement_seq,user_id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,origin_stop_code,destination_stop_code,origin_municipality,destination_municipality,diseno777subida,diseno777bajada,origin_time_utc,destination_time_utc,periodosubida,periodobajada,day_type,mediahora,contrato,trip_weight,tiempomediodeviaje,time_period,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,purpose,dviaje_buses,etapas_detectadas,linea_metro_subida,zona777subida,linea_metro_bajada,zona777bajada,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,m0,0,4306298,1,2,NaN,0,1,3098.0000,51.6333,15836.0068,25695.0000,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,172,299,2019-05-13 10:00:55+00:00,2019-05-13 10:52:33+00:00,03 - TRANSICION NOCTURNO,04 - PUNTA MANANA,weekday,06:00:00,NaN,1.4376,2019-05-13 06:26:44,night,06:00:00,LABORAL,NaN,48.8667,SM2H,work,NaN,2,L5,172,L5,299,-70.783226,-33.524734,-70.626419,-33.467472,88b2c540ddfffff,88b2c5548bfffff
1,m1,m1,0,8398026,2,1,NaN,0,1,701.0000,11.6833,3872.8728,4084.0000,T-18-117-NS-5,T-18-153-PO-35,NUNOA,NUNOA,231,240,2019-05-13 10:53:45+00:00,2019-05-13 11:05:26+00:00,04 - PUNTA MANANA,04 - PUNTA MANANA,weekday,06:30:00,NaN,1.2547,2019-05-13 06:59:35,morning,06:30:00,LABORAL,NaN,11.6833,SM2H,work,NaN,1,<NA>,231,<NA>,240,-70.629251,-33.452086,-70.587011,-33.457846,88b2c554c3fffff,88b2c50b29fffff
2,m2,m2,0,9173978,1,1,NaN,0,1,690.0000,11.5000,4511.1367,4513.0000,SANTA LUCIA,SAN ALBERTO HURTADO,SANTIAGO,ESTACION CENTRAL,286,77,2019-05-13 22:12:35+00:00,2019-05-13 22:24:05+00:00,09 - PUNTA TARDE,09 - PUNTA TARDE,weekday,18:00:00,NaN,1.2976,2019-05-13 18:18:20,afternoon,18:00:00,LABORAL,NaN,11.5000,MS_M_M,other,NaN,1,L1,286,L1,77,-70.645817,-33.442849,-70.692442,-33.454116,88b2c55413fffff,88b2c555cbfffff
3,m3,m3,0,9284970,1,1,NaN,0,1,1209.0000,20.1500,3875.6917,5234.0000,T-17-149-SN-8,T-17-140-OP-35,LAS CONDES,LAS CONDES,775,207,2019-05-13 10:56:49+00:00,2019-05-13 11:16:58+00:00,04 - PUNTA MANANA,04 - PUNTA MANANA,weekday,06:30:00,NaN,1.4500,2019-05-13 07:06:53,morning,07:00:00,LABORAL,NaN,20.1500,SM2H,work,NaN,1,<NA>,775,<NA>,207,-70.529608,-33.420266,-70.569059,-33.409114,88b2c51995fffff,88b2c519adfffff
4,m4,m4,0,9403002,1,1,NaN,0,1,1320.0000,22.0000,9128.2852,9258.0000,ESCUELA MILITAR,UNION LATINO AMERICANA,LAS CONDES,SANTIAGO,215,280,2019-05-13 20:08:12+00:00,2019-05-13 20:30:12+00:00,08 - FUERA DE PUNTA TARDE,08 - FUERA DE PUNTA TARDE,weekday,16:00:00,NaN,1.2666,2019-05-13 16:19:12,afternoon,16:00:00,LABORAL,NaN,22.0000,M3B,other,NaN,1,L1,215,L1,280,-70.584445,-33.414320,-70.673213,-33.449471,88b2c556d3fffff,88b2c5543bfffff


[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'mode' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='mode', source_field=None, row_count=None, details={'field': 'mode', 'source_columns': ['id', 'nviaje', 'netapa', 'etapas', 'netapassinbajada', 'ultimaetapaconbajada', 'tviaje_seg', 'tviaje_min', 'dviajeeuclidiana_mts', 'dviajeenruta_mts', 'paraderosubida', 'paraderobajada', 'comunasubida', 'comunabajada', 'diseno777subida', 'diseno777bajada', 'tiemposubida', 'tiempobajada', 'periodosubida', 'periodobajada', 'tipodia', 'mediahora', 'contrato', 'factorexpansion', 'tiempomediodeviaje', 'periodomediodeviaje', 'mediahoramediodeviaje', 'tipodiamediodeviaje', 'escolar', 'tviaje_en_vehiculo_min', 'tipo_corte_etapa_viaje', 'proposito', 'dviaje_buses', 'etapas_detectadas', 'linea_metro_subida', 'zona777subida', 'linea_metro_bajada', 'zona777bajada', 'subida_lon', 'subida_lat', 'bajada_lon', 'bajada_lat'], 'field_cor

### 2) Uso más personalizado, con decisiones específicas

In [5]:
from scripts.source_profiles.factories_adatrap.trips_defaults import (
    make_adatrap_trips_default_schema,
    make_adatrap_trips_default_value_correspondence,
    load_yaml_file,
    clean_domain_dict,
    build_adatrap_time_period_mapping,
)
from scripts.source_profiles.factories_adatrap.make_adatrap_trips_profile import _merge_field_correspondence
from pylondrina.importing import ImportOptions
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.types import FieldCorrespondence, ValueCorrespondence

# -----------------------------------------------------------------------------
# Objetos para caso personalizado
# -----------------------------------------------------------------------------

def make_adatrap_trips_custom_schema(adatrap_domains_yaml: str | Path) -> TripSchema:
    base = make_adatrap_trips_default_schema(adatrap_domains_yaml)

    selected = {
        k: v
        for k, v in base.fields.items()
        if k in {
            "movement_id",
            "user_id",
            "origin_longitude",
            "origin_latitude",
            "destination_longitude",
            "destination_latitude",
            "origin_h3_index",
            "destination_h3_index",
            "trip_id",
            "movement_seq",
            "origin_time_utc",
            "destination_time_utc",
            "origin_municipality",
            "destination_municipality",
            "origin_stop_code",
            "destination_stop_code",
            "purpose",
            "day_type",
            "time_period",
            "trip_weight",
        }
    }

    return TripSchema(
        version="1.1-adatrap-custom",
        fields=selected,
        required=[
            "movement_id",
            "user_id",
            "origin_longitude",
            "origin_latitude",
            "destination_longitude",
            "destination_latitude",
            "origin_h3_index",
            "destination_h3_index",
            "trip_id",
            "movement_seq",
        ],
    )

ADATRAP_TRIPS_CUSTOM_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
        "origin_time_utc",
        "destination_time_utc",
        "origin_municipality",
        "destination_municipality",
        "origin_stop_code",
        "destination_stop_code",
        "purpose",
        "day_type",
        "time_period",
        "trip_weight",
    ],
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone="America/Santiago",
)

ADATRAP_TRIPS_CUSTOM_FIELD_CORRESPONDENCE: FieldCorrespondence = {
    "user_id": "id",
    "origin_longitude": "subida_lon",
    "origin_latitude": "subida_lat",
    "destination_longitude": "bajada_lon",
    "destination_latitude": "bajada_lat",
    "origin_time_utc": "tiemposubida",
    "destination_time_utc": "tiempobajada",
    "origin_municipality": "comunasubida",
    "destination_municipality": "comunabajada",
    "origin_stop_code": "paraderosubida",
    "destination_stop_code": "paraderobajada",
    "purpose": "proposito",
    "day_type": "tipodia",
    "time_period": "periodosubida",
    "trip_weight": "factorexpansion",
}

def make_adatrap_trips_custom_value_correspondence(
    adatrap_domains_yaml: str | Path,
) -> ValueCorrespondence:
    base = make_adatrap_trips_default_value_correspondence(adatrap_domains_yaml)
    base["purpose"]["SINBAJADA"] = "other"

    domains_raw = load_yaml_file(adatrap_domains_yaml)
    domains = clean_domain_dict(domains_raw)
    base["time_period"] = build_adatrap_time_period_mapping(domains["periodosubida"])

    return base

ADATRAP_TRIPS_CUSTOM_PROVENANCE_EXAMPLE = {
    "source": {
        "name": "ADATRAP",
        "profile": "ADATRAP_TRIPS_CUSTOM",
        "entity": "trips",
        "version": "perfil_semana",
    },
    "notes": [
        "factory nivel 3 ADATRAP trips custom",
        "time_period usa periodosubida",
    ],
}

In [7]:
custom_schema = make_adatrap_trips_custom_schema(DATA_PATH / "domains.yaml")
custom_value_corr = make_adatrap_trips_custom_value_correspondence(DATA_PATH / "domains.yaml")

profile_custom = make_adatrap_trips_profile(
    df_stops=adatrap_stops,
    stage_layout_yaml=DATA_PATH / "stage_layout.yaml",
    domains_yaml=DATA_PATH / "domains.yaml",
    schema_override=custom_schema,
    value_correspondence_override=custom_value_corr,
    options_override={
        "keep_extra_fields": ADATRAP_TRIPS_CUSTOM_OPTIONS.keep_extra_fields,
        "selected_fields": ADATRAP_TRIPS_CUSTOM_OPTIONS.selected_fields,
        "strict": ADATRAP_TRIPS_CUSTOM_OPTIONS.strict,
        "strict_domains": ADATRAP_TRIPS_CUSTOM_OPTIONS.strict_domains,
        "single_stage": ADATRAP_TRIPS_CUSTOM_OPTIONS.single_stage,
        "source_timezone": ADATRAP_TRIPS_CUSTOM_OPTIONS.source_timezone,
    },
    profile_name="ADATRAP_TRIPS_CUSTOM",
)


# inspección clara
#profile_custom.schema_override
#profile_custom.default_options
display(profile_custom.default_field_correspondence)
#profile_custom.default_value_correspondence

{'user_id': 'id',
 'origin_longitude': 'subida_lon',
 'origin_latitude': 'subida_lat',
 'destination_longitude': 'bajada_lon',
 'destination_latitude': 'bajada_lat',
 'origin_time_utc': 'tiemposubida',
 'destination_time_utc': 'tiempobajada',
 'origin_municipality': 'comunasubida',
 'destination_municipality': 'comunabajada',
 'origin_stop_code': 'paraderosubida',
 'destination_stop_code': 'paraderobajada',
 'purpose': 'proposito',
 'day_type': 'tipodia',
 'time_period': 'periodomediodeviaje',
 'trip_weight': 'factorexpansion'}

In [8]:
trips_adatrap_custom, report_adatrap_custom = import_trips_from_profile(
    profile=profile_custom,
    df=adatrap_viajes.head(5000),
    field_correspondence=ADATRAP_TRIPS_CUSTOM_FIELD_CORRESPONDENCE,
    source_name="ADATRAP trips level-3 custom factory",
    provenance=ADATRAP_TRIPS_CUSTOM_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)

display(trips_adatrap_custom.data.head())
display(report_adatrap_custom.issues)

,movement_id,trip_id,movement_seq,user_id,origin_stop_code,destination_stop_code,origin_municipality,destination_municipality,origin_time_utc,destination_time_utc,time_period,day_type,trip_weight,purpose,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,m0,0,4306298,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,2019-05-13 10:00:55+00:00,2019-05-13 10:52:33+00:00,night,weekday,1.4376,work,-70.783226,-33.524734,-70.626419,-33.467472,88b2c540ddfffff,88b2c5548bfffff
1,m1,m1,0,8398026,T-18-117-NS-5,T-18-153-PO-35,NUNOA,NUNOA,2019-05-13 10:53:45+00:00,2019-05-13 11:05:26+00:00,morning,weekday,1.2547,work,-70.629251,-33.452086,-70.587011,-33.457846,88b2c554c3fffff,88b2c50b29fffff
2,m2,m2,0,9173978,SANTA LUCIA,SAN ALBERTO HURTADO,SANTIAGO,ESTACION CENTRAL,2019-05-13 22:12:35+00:00,2019-05-13 22:24:05+00:00,afternoon,weekday,1.2976,other,-70.645817,-33.442849,-70.692442,-33.454116,88b2c55413fffff,88b2c555cbfffff
3,m3,m3,0,9284970,T-17-149-SN-8,T-17-140-OP-35,LAS CONDES,LAS CONDES,2019-05-13 10:56:49+00:00,2019-05-13 11:16:58+00:00,morning,weekday,1.4500,work,-70.529608,-33.420266,-70.569059,-33.409114,88b2c51995fffff,88b2c519adfffff
4,m4,m4,0,9403002,ESCUELA MILITAR,UNION LATINO AMERICANA,LAS CONDES,SANTIAGO,2019-05-13 20:08:12+00:00,2019-05-13 20:30:12+00:00,afternoon,weekday,1.2666,other,-70.584445,-33.414320,-70.673213,-33.449471,88b2c556d3fffff,88b2c5543bfffff


[Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'destination_time_utc' falló en algunas filas (13.2%); se marcarán como nulos para validación posterior.", field='destination_time_utc', source_field=None, row_count=662, details={'field': 'destination_time_utc', 'dtype_expected': 'datetime', 'parse_fail_count': 662, 'total_count': 5000, 'fail_rate': 0.1324, 'fallback': 'set_null', 'action': 'continue'}),
 Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'destination_time_utc' falló en algunas filas (13.2%); se marcarán como nulos para validación posterior.", field='destination_time_utc', source_field=None, row_count=662, details={'field': 'destination_time_utc', 'dtype_expected': 'datetime', 'parse_fail_count': 662, 'total_count': 5000, 'fail_rate': 0.1324, 'fallback': 'set_null', 'action': 'continue'}),
 Issue(level='warning', code='IMP.H3.PARTIAL_DERIVATION', message='La derivación de índices 

## Para datos del sistema ADATRAP - Stages / etapas

### 1) Uso normal, rápido, sin decisiones de schema/campos/valores en la factory

In [3]:
from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_adatrap.make_adatrap_stages_profile import make_adatrap_stages_profile
from scripts.source_profiles.factories_adatrap.stages_defaults import (
    ADATRAP_STAGES_DEFAULT_PROVENANCE_EXAMPLE,
)

adatrap_viajes = pd.read_csv(
    DATA_PATH / "semana_2019_05_13.viajes",
    sep="|",
    dtype=str,
    low_memory=False,
)
adatrap_stops = pd.read_csv(
    DATA_PATH / "DIC_777_fixed.csv",
    sep=",",
    dtype=str,
    low_memory=False,
)

profile = make_adatrap_stages_profile(
    df_stops=adatrap_stops,
    stage_layout_yaml=DATA_PATH / "stage_layout.yaml",
    domains_yaml=DATA_PATH / "domains.yaml",
)

# inspección clara
#profile.schema_override
#profile.default_options
#profile.default_field_correspondence
#profile.default_value_correspondence

In [4]:
df_test = profile.preprocess(adatrap_viajes.head(1000))
display(df_test.head())

,id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,paraderosubida,paraderobajada,comunasubida,comunabajada,diseno777subida,diseno777bajada,tiemposubida,tiempobajada,periodosubida,periodobajada,tipodia,mediahora,contrato,factorexpansion,tiempomediodeviaje,periodomediodeviaje,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,proposito,dviaje_buses,numero_etapa,t_etapa,d_etapa,tespera_etapa,ttrasbordo_etapa,tcaminata_etapa,dtransbordo_etapa,op_etapa,tipoop_etapa,serv_etapa,linea_metro_subida_etapa,linea_metro_bajada_etapa,paraderosubida_etapa,tiemposubida_etapa,zona777subida_etapa,paraderobajada_etapa,tiempobajada_etapa,zona777bajada_etapa,tipotransporte_etapa,tesperaest_etapa,subida_lon,subida_lat,bajada_lon,bajada_lat
0,4306298,1,2,-,0,1,3098.0000,51.6333,15836.0068,25695.0000,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,172,299,2019-05-13 06:00:55,2019-05-13 06:52:33,03 - TRANSICION NOCTURNO,04 - PUNTA MANANA,LABORAL,06:00:00,-,1.4376,2019-05-13 06:26:44,03 - TRANSICION NOCTURNO,06:00:00,LABORAL,-,48.8667,SM2H,TRABAJO,-,1,10.6333,4433.0000,1.5852,2.7667,1.1814,106.3297,4,-,T401 00I,-,-,T-13-104-PO-15,2019-05-13 06:00:55,172,E-13-54-SN-10,2019-05-13 06:11:33,819,BUS,-,-70.783226,-33.524734,-70.757084,-33.509308
1,4306298,1,2,-,0,1,3098.0000,51.6333,15836.0068,25695.0000,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,172,299,2019-05-13 06:00:55,2019-05-13 06:52:33,03 - TRANSICION NOCTURNO,04 - PUNTA MANANA,LABORAL,06:00:00,-,1.4376,2019-05-13 06:26:44,03 - TRANSICION NOCTURNO,06:00:00,LABORAL,-,48.8667,SM2H,TRABAJO,-,2,38.2333,21262.0000,-,-,-,-,-,-,L5,L5,L5,PLAZA MAIPU,2019-05-13 06:14:19,819,NUBLE,2019-05-13 06:52:33,299,METRO,-,-70.756241,-33.510222,-70.626419,-33.467472
2,8398026,2,1,-,0,1,701.0000,11.6833,3872.8728,4084.0000,T-18-117-NS-5,T-18-153-PO-35,NUNOA,NUNOA,231,240,2019-05-13 06:53:45,2019-05-13 07:05:26,04 - PUNTA MANANA,04 - PUNTA MANANA,LABORAL,06:30:00,-,1.2547,2019-05-13 06:59:35,04 - PUNTA MANANA,06:30:00,LABORAL,-,11.6833,SM2H,TRABAJO,-,1,11.6833,4084.0000,-,-,-,-,5,-,T513 01I,-,-,T-18-117-NS-5,2019-05-13 06:53:45,231,T-18-153-PO-35,2019-05-13 07:05:26,240,BUS,-,-70.629251,-33.452086,-70.587011,-33.457846
3,9173978,1,1,-,0,1,690.0000,11.5000,4511.1367,4513.0000,SANTA LUCIA,SAN ALBERTO HURTADO,SANTIAGO,ESTACION CENTRAL,286,77,2019-05-13 18:12:35,2019-05-13 18:24:05,09 - PUNTA TARDE,09 - PUNTA TARDE,LABORAL,18:00:00,-,1.2976,2019-05-13 18:18:20,09 - PUNTA TARDE,18:00:00,LABORAL,-,11.5000,MS_M_M,OTROS,-,1,11.5000,4513.0000,-,-,-,-,-,-,L1,L1,L1,SANTA LUCIA,2019-05-13 18:12:35,286,SAN ALBERTO HURTADO,2019-05-13 18:24:05,77,METRO,-,-70.645817,-33.442849,-70.692442,-33.454116
4,9284970,1,1,-,0,1,1209.0000,20.1500,3875.6917,5234.0000,T-17-149-SN-8,T-17-140-OP-35,LAS CONDES,LAS CONDES,775,207,2019-05-13 06:56:49,2019-05-13 07:16:58,04 - PUNTA MANANA,04 - PUNTA MANANA,LABORAL,06:30:00,-,1.4500,2019-05-13 07:06:53,04 - PUNTA MANANA,07:00:00,LABORAL,-,20.1500,SM2H,TRABAJO,-,1,20.1500,5234.0000,-,-,-,-,4,-,T407 00R,-,-,T-17-149-SN-8,2019-05-13 06:56:49,775,T-17-140-OP-35,2019-05-13 07:16:58,207,BUS,-,-70.529608,-33.420266,-70.569059,-33.409114


In [5]:
stages_adatrap, report_stages_adatrap = import_trips_from_profile(
    profile=profile,
    df=adatrap_viajes.head(5000),
    source_name="ADATRAP stages level-3 factory",
    provenance=ADATRAP_STAGES_DEFAULT_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)
display(stages_adatrap.data)
display(report_stages_adatrap.issues)

,movement_id,user_id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,paraderosubida,paraderobajada,origin_municipality,destination_municipality,diseno777subida,diseno777bajada,tiemposubida,tiempobajada,periodosubida,periodobajada,day_type,mediahora,contrato,trip_weight,tiempomediodeviaje,time_period,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,purpose,dviaje_buses,movement_seq,t_etapa,d_etapa,tespera_etapa,ttrasbordo_etapa,tcaminata_etapa,dtransbordo_etapa,op_etapa,tipoop_etapa,serv_etapa,linea_metro_subida_etapa,linea_metro_bajada_etapa,origin_stop_code,origin_time_utc,zona777subida_etapa,destination_stop_code,destination_time_utc,zona777bajada_etapa,mode,tesperaest_etapa,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,4306298,1,2,-,0,1,3098.0000,51.6333,15836.0068,25695.0000,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,172,299,2019-05-13 06:00:55,2019-05-13 06:52:33,03 - TRANSICION NOCTURNO,04 - PUNTA MANANA,weekday,06:00:00,-,1.4376,2019-05-13 06:26:44,night,06:00:00,LABORAL,-,48.8667,SM2H,work,-,1,10.6333,4433.0000,1.5852,2.7667,1.1814,106.3297,4,-,T401 00I,-,-,T-13-104-PO-15,2019-05-13 10:00:55+00:00,172,E-13-54-SN-10,2019-05-13 10:11:33+00:00,819,bus,-,-70.783226,-33.524734,-70.757084,-33.509308,88b2c540ddfffff,88b2c542bbfffff
1,m1,4306298,1,2,-,0,1,3098.0000,51.6333,15836.0068,25695.0000,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,172,299,2019-05-13 06:00:55,2019-05-13 06:52:33,03 - TRANSICION NOCTURNO,04 - PUNTA MANANA,weekday,06:00:00,-,1.4376,2019-05-13 06:26:44,night,06:00:00,LABORAL,-,48.8667,SM2H,work,-,2,38.2333,21262.0000,-,-,-,-,-,-,L5,L5,L5,PLAZA MAIPU,2019-05-13 10:14:19+00:00,819,NUBLE,2019-05-13 10:52:33+00:00,299,metro,-,-70.756241,-33.510222,-70.626419,-33.467472,88b2c54297fffff,88b2c5548bfffff
2,m2,8398026,2,1,-,0,1,701.0000,11.6833,3872.8728,4084.0000,T-18-117-NS-5,T-18-153-PO-35,NUNOA,NUNOA,231,240,2019-05-13 06:53:45,2019-05-13 07:05:26,04 - PUNTA MANANA,04 - PUNTA MANANA,weekday,06:30:00,-,1.2547,2019-05-13 06:59:35,morning,06:30:00,LABORAL,-,11.6833,SM2H,work,-,1,11.6833,4084.0000,-,-,-,-,5,-,T513 01I,-,-,T-18-117-NS-5,2019-05-13 10:53:45+00:00,231,T-18-153-PO-35,2019-05-13 11:05:26+00:00,240,bus,-,-70.629251,-33.452086,-70.587011,-33.457846,88b2c554c3fffff,88b2c50b29fffff
3,m3,9173978,1,1,-,0,1,690.0000,11.5000,4511.1367,4513.0000,SANTA LUCIA,SAN ALBERTO HURTADO,SANTIAGO,ESTACION CENTRAL,286,77,2019-05-13 18:12:35,2019-05-13 18:24:05,09 - PUNTA TARDE,09 - PUNTA TARDE,weekday,18:00:00,-,1.2976,2019-05-13 18:18:20,afternoon,18:00:00,LABORAL,-,11.5000,MS_M_M,other,-,1,11.5000,4513.0000,-,-,-,-,-,-,L1,L1,L1,SANTA LUCIA,2019-05-13 22:12:35+00:00,286,SAN ALBERTO HURTADO,2019-05-13 22:24:05+00:00,77,metro,-,-70.645817,-33.442849,-70.692442,-33.454116,88b2c55413fffff,88b2c555cbfffff
4,m4,9284970,1,1,-,0,1,1209.0000,20.1500,3875.6917,5234.0000,T-17-149-SN-8,T-17-140-OP-35,LAS CONDES,LAS CONDES,775,207,2019-05-13 06:56:49,2019-05-13 07:16:58,04 - PUNTA MANANA,04 - PUNTA MANANA,weekday,06:30:00,-,1.4500,2019-05-13 07:06:53,morning,07:00:00,LABORAL,-,20.1500,SM2H,work,-,1,20.1500,5234.0000,-,-,-,-,4,-,T407 00R,-,-,T-17-149-SN-8,2019-05-13 10:56:49+00:00,775,T-17-140-OP-35,2019-05-13 11:16:58+00:00,207,bus,-,-70.529608,-33.420266,-70.569059,-33.409114,88b2c51995fffff,88b2c519adfffff
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6790,m6790,35122498,1,2,-,0,1,912.0000,15.2000,6040.3154,6955.0000,L-24-1-6-SN,VICUNA MACKENNA,SAN RAMON,LA FLORIDA,354,470,2019-05-13 10:53:45,2019-05-13 11:08:57,06 - FUERA DE PUNTA MANANA,06 - FUERA DE PUNTA MANANA,weekday,10:30:00,-,1.4224,2019-05-13 11:01:21,morning,11:00:00,LABORAL,-,12.3167,SM2H,work,-,2,8.2333,5500.0000

[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'user_gender' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='user_gender', source_field=None, row_count=None, details={'field': 'user_gender', 'source_columns': ['id', 'nviaje', 'netapa', 'etapas', 'netapassinbajada', 'ultimaetapaconbajada', 'tviaje_seg', 'tviaje_min', 'dviajeeuclidiana_mts', 'dviajeenruta_mts', 'paraderosubida', 'paraderobajada', 'comunasubida', 'comunabajada', 'diseno777subida', 'diseno777bajada', 'tiemposubida', 'tiempobajada', 'periodosubida', 'periodobajada', 'tipodia', 'mediahora', 'contrato', 'factorexpansion', 'tiempomediodeviaje', 'periodomediodeviaje', 'mediahoramediodeviaje', 'tipodiamediodeviaje', 'escolar', 'tviaje_en_vehiculo_min', 'tipo_corte_etapa_viaje', 'proposito', 'dviaje_buses', 'numero_etapa', 't_etapa', 'd_etapa', 'tespera_etapa', 'ttrasbordo_etapa', 'tcaminata_etapa', 'dtransbordo_etapa', 'op_etapa', 'tipoop_etapa', 's

### 2) Uso más personalizado, con decisiones específicas

In [13]:
from scripts.source_profiles.factories_adatrap.trips_defaults import (
    load_yaml_file,
    clean_domain_dict,
    build_adatrap_time_period_mapping,
)

from pylondrina.importing import ImportOptions
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.types import FieldCorrespondence, ValueCorrespondence

# -------------------------------------------------------------------------
# Objetos custom independientes de los defaults
# -------------------------------------------------------------------------

def make_adatrap_stages_custom_schema(adatrap_domains_yaml: str | Path) -> TripSchema:
    domains_raw = load_yaml_file(adatrap_domains_yaml)
    domains = clean_domain_dict(domains_raw)

    return TripSchema(
        version="1.1-adatrap-stages-custom",
        fields={
            "movement_id": FieldSpec("movement_id", "string", required=True),
            "user_id": FieldSpec("user_id", "string", required=True),
            "movement_seq": FieldSpec("movement_seq", "int", required=True),

            "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
            "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
            "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
            "destination_latitude": FieldSpec("destination_latitude", "float", required=True),
            "origin_h3_index": FieldSpec("origin_h3_index", "string", required=True),
            "destination_h3_index": FieldSpec("destination_h3_index", "string", required=True),

            "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
            "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),
            "origin_stop_code": FieldSpec("origin_stop_code", "string", required=False),
            "destination_stop_code": FieldSpec("destination_stop_code", "string", required=False),
            "origin_municipality": FieldSpec("origin_municipality", "string", required=False),
            "destination_municipality": FieldSpec("destination_municipality", "string", required=False),

            "mode": FieldSpec(
                "mode",
                "categorical",
                required=False,
                domain=DomainSpec(values=["bus", "metro", "train", "other"], extendable=True),
            ),
            "purpose": FieldSpec(
                "purpose",
                "categorical",
                required=False,
                domain=DomainSpec(values=["home", "work", "other"], extendable=True),
            ),
            "day_type": FieldSpec(
                "day_type",
                "categorical",
                required=False,
                domain=DomainSpec(values=["weekday", "weekend"], extendable=True),
            ),
            "time_period": FieldSpec(
                "time_period",
                "categorical",
                required=False,
                domain=DomainSpec(
                    values=[],
                    extendable=True,
                ),
            ),
            "trip_weight": FieldSpec("trip_weight", "float", required=False),
        },
        required=[
            "movement_id",
            "user_id",
            "movement_seq",
            "origin_longitude",
            "origin_latitude",
            "destination_longitude",
            "destination_latitude",
            "origin_h3_index",
            "destination_h3_index",
        ],
    )

ADATRAP_STAGES_CUSTOM_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=[
        "movement_id",
        "user_id",
        "movement_seq",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "origin_time_utc",
        "destination_time_utc",
        "origin_stop_code",
        "destination_stop_code",
        "origin_municipality",
        "destination_municipality",
        "mode",
        "purpose",
        "day_type",
        "time_period",
        "trip_weight",
    ],
    strict=False,
    strict_domains=False,
    single_stage=False,
    source_timezone="America/Santiago",
)

ADATRAP_STAGES_CUSTOM_FIELD_CORRESPONDENCE: FieldCorrespondence = {
    "user_id": "id",
    "movement_seq": "numero_etapa",

    "origin_longitude": "subida_lon",
    "origin_latitude": "subida_lat",
    "destination_longitude": "bajada_lon",
    "destination_latitude": "bajada_lat",

    "origin_time_utc": "tiemposubida_etapa",
    "destination_time_utc": "tiempobajada_etapa",

    "origin_municipality": "comunasubida",
    "destination_municipality": "comunabajada",
    "origin_stop_code": "paraderosubida_etapa",
    "destination_stop_code": "paraderobajada_etapa",

    "mode": "tipotransporte_etapa",
    "purpose": "proposito",
    "day_type": "tipodia",
    "time_period": "periodomediodeviaje",
    "trip_weight": "factorexpansion",
}

def make_adatrap_stages_custom_value_correspondence(
    adatrap_domains_yaml: str | Path,
) -> ValueCorrespondence:
    domains_raw = load_yaml_file(adatrap_domains_yaml)
    domains = clean_domain_dict(domains_raw)

    return {
        "mode": {
            "BUS": "bus",
            "METRO": "metro",
            "METROTREN": "train",
            "ZP": "other",
        },
        "purpose": {
            "HOGAR": "home",
            "TRABAJO": "work",
            "OTROS": "other",
            "MENOS1MINUTO": "other",
            "SINBAJADA": "other",
        },
        "day_type": {
            "LABORAL": "weekday",
            "SABADO": "weekend",
            "DOMINGO": "weekend",
        },
        "time_period": build_adatrap_time_period_mapping(domains["periodomediodeviaje"]),
    }

ADATRAP_STAGES_CUSTOM_PROVENANCE_EXAMPLE = {
    "source": {
        "name": "ADATRAP",
        "profile": "ADATRAP_STAGES_CUSTOM",
        "entity": "stages",
        "version": "perfil_semana",
    },
    "notes": [
        "factory nivel 3 ADATRAP stages custom",
        "schema y mappings definidos explícitamente sin depender de defaults",
    ],
}

In [14]:
custom_schema = make_adatrap_stages_custom_schema(DATA_PATH / "domains.yaml")
custom_value_corr = make_adatrap_stages_custom_value_correspondence(DATA_PATH / "domains.yaml")

profile_custom = make_adatrap_stages_profile(
    df_stops=adatrap_stops,
    stage_layout_yaml=DATA_PATH / "stage_layout.yaml",
    domains_yaml=DATA_PATH / "domains.yaml",
    schema_override=custom_schema,
    field_correspondence_override=ADATRAP_STAGES_CUSTOM_FIELD_CORRESPONDENCE,
    value_correspondence_override=custom_value_corr,
    options_override={
        "keep_extra_fields": ADATRAP_STAGES_CUSTOM_OPTIONS.keep_extra_fields,
        "selected_fields": ADATRAP_STAGES_CUSTOM_OPTIONS.selected_fields,
        "strict": ADATRAP_STAGES_CUSTOM_OPTIONS.strict,
        "strict_domains": ADATRAP_STAGES_CUSTOM_OPTIONS.strict_domains,
        "single_stage": ADATRAP_STAGES_CUSTOM_OPTIONS.single_stage,
        "source_timezone": ADATRAP_STAGES_CUSTOM_OPTIONS.source_timezone,
    },
    profile_name="ADATRAP_STAGES_CUSTOM",
)


# inspección clara
display(profile_custom.schema_override)
profile_custom.default_options
profile_custom.default_field_correspondence
profile_custom.default_value_correspondence


{'version': '1.1-adatrap-stages-custom',
 'required': ['movement_id',
              'user_id',
              'movement_seq',
              'origin_longitude',
              'origin_latitude',
              'destination_longitude',
              'destination_latitude',
              'origin_h3_index',
              'destination_h3_index'],
 'semantic_rules': None,
 'fields': {'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': True,
                            'constraints': None,
                            'domain': None},
            'user_id': {'name': 'user_id', 'dtype': 'string', 'required': True, 'constraints': None, 'domain': None},
            'movement_seq': {'name': 'movement_seq',
                             'dtype': 'int',
                             'required': True,
                             'constraints': None,
                             'domain': None},
            'origin_longitude': {'name': 'origin_longitude',
                                 'dtype': 'float',
                                 'required': True,
                                 'constraints': None,
                                 'domain': None},
            'origin_latitude': {'name': 'origin_latitude',
                                'dtype': 'float',
                                'required': True,
                                'constraints': None,
                                'domain': None},
            'destination_longitude': {'name': 'destination_longitude',
                                      'dtype': 'float',
                                      'required': True,
                                      'constraints': None,
                                      'domain': None},
            'destination_latitude': {'name': 'destination_latitude',
                                     'dtype': 'float',
                                     'required': True,
                                     'constraints': None,
                                     'domain': None},
            'origin_h3_index': {'name': 'origin_h3_index',
                                'dtype': 'string',
                                'required': True,
                                'constraints': None,
                                'domain': None},
            'destination_h3_index': {'name': 'destination_h3_index',
                                     'dtype': 'string',
                                     'required': True,
                                     'constraints': None,
                                     'domain': None},
            'origin_time_utc': {'name': 'origin_time_utc',
                                'dtype': 'datetime',
                                'required': False,
                                'constraints': None,
                                'domain': None},
            'destination_time_utc': {'name': 'destination_time_utc',
                                     'dtype': 'datetime',
                                     'required': False,
                                     'constraints': None,
                                     'domain': None},
            'origin_stop_code': {'name': 'origin_stop_code',
                                 'dtype': 'string',
                                 'required': False,
                                 'constraints': None,
                                 'domain': None},
            'destination_stop_code': {'name': 'destination_stop_code',
                                      'dtype': 'string',
                                      'required': False,
                                      'constraints': None,
                                      'domain': None},
            'origin_municipality': {'name': 'origin_municipality',
                                    'dtype': 'string',
                                    'required': False,
                                    'constraints': None,
     

{'mode': {'BUS': 'bus', 'METRO': 'metro', 'METROTREN': 'train', 'ZP': 'other'},
 'purpose': {'HOGAR': 'home',
  'TRABAJO': 'work',
  'OTROS': 'other',
  'MENOS1MINUTO': 'other',
  'SINBAJADA': 'other'},
 'day_type': {'LABORAL': 'weekday', 'SABADO': 'weekend', 'DOMINGO': 'weekend'},
 'time_period': {'01 - PRE NOCTURNO': 'evening',
  '01 - PRE NOCTURNO DOMINGO': 'evening',
  '01 - PRE NOCTURNO SABADO': 'evening',
  '02 - NOCTURNO': 'night',
  '02 - NOCTURNO DOMINGO': 'night',
  '02 - NOCTURNO SABADO': 'night',
  '03 - TRANSICION DOMINGO MANANA': 'morning',
  '03 - TRANSICION NOCTURNO': 'night',
  '03 - TRANSICION SABADO MANANA': 'morning',
  '04 - MANANA DOMINGO': 'morning',
  '04 - PUNTA MANANA': 'morning',
  '04 - PUNTA MANANA SABADO': 'morning',
  '05 - MANANA SABADO': 'morning',
  '05 - MEDIODIA DOMINGO': 'midday',
  '05 - TRANSICION PUNTA MANANA': 'morning',
  '06 - FUERA DE PUNTA MANANA': 'morning',
  '06 - PUNTA MEDIODIA SABADO': 'midday',
  '06 - TARDE DOMINGO': 'afternoon',
  '0

In [15]:
stages_adatrap_custom, report_stages_adatrap_custom = import_trips_from_profile(
    profile=profile_custom,
    df=adatrap_viajes.head(5000),
    source_name="ADATRAP stages level-3 custom factory",
    provenance=ADATRAP_STAGES_CUSTOM_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)
display(stages_adatrap_custom.data)
display(report_stages_adatrap_custom.issues)

,movement_id,user_id,origin_municipality,destination_municipality,day_type,trip_weight,time_period,purpose,movement_seq,origin_stop_code,origin_time_utc,destination_stop_code,destination_time_utc,mode,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,4306298,MAIPU,NUNOA,weekday,1.4376,night,work,1,T-13-104-PO-15,2019-05-13 10:00:55+00:00,E-13-54-SN-10,2019-05-13 10:11:33+00:00,bus,-70.783226,-33.524734,-70.757084,-33.509308,88b2c540ddfffff,88b2c542bbfffff
1,m1,4306298,MAIPU,NUNOA,weekday,1.4376,night,work,2,PLAZA MAIPU,2019-05-13 10:14:19+00:00,NUBLE,2019-05-13 10:52:33+00:00,metro,-70.756241,-33.510222,-70.626419,-33.467472,88b2c54297fffff,88b2c5548bfffff
2,m2,8398026,NUNOA,NUNOA,weekday,1.2547,morning,work,1,T-18-117-NS-5,2019-05-13 10:53:45+00:00,T-18-153-PO-35,2019-05-13 11:05:26+00:00,bus,-70.629251,-33.452086,-70.587011,-33.457846,88b2c554c3fffff,88b2c50b29fffff
3,m3,9173978,SANTIAGO,ESTACION CENTRAL,weekday,1.2976,afternoon,other,1,SANTA LUCIA,2019-05-13 22:12:35+00:00,SAN ALBERTO HURTADO,2019-05-13 22:24:05+00:00,metro,-70.645817,-33.442849,-70.692442,-33.454116,88b2c55413fffff,88b2c555cbfffff
4,m4,9284970,LAS CONDES,LAS CONDES,weekday,1.4500,morning,work,1,T-17-149-SN-8,2019-05-13 10:56:49+00:00,T-17-140-OP-35,2019-05-13 11:16:58+00:00,bus,-70.529608,-33.420266,-70.569059,-33.409114,88b2c51995fffff,88b2c519adfffff
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6790,m6790,35122498,SAN RAMON,LA FLORIDA,weekday,1.4224,morning,work,2,SAN RAMON,2019-05-13 15:00:43+00:00,VICUNA MACKENNA,2019-05-13 15:08:57+00:00,metro,-70.642469,-33.541309,-70.596074,-33.519649,88b2c546e9fffff,88b2c50939fffff
6791,m6791,35124770,PUENTE ALTO,-,weekday,0.0000,-,other,1,L-34-70-95-OP,2019-05-13 17:45:54+00:00,-,NaT,bus,-70.562415,-33.596680,NaN,NaN,88b2c57319fffff,<NA>
6792,m6792,35127650,SANTIAGO,SANTIAGO,weekday,1.3580,night,home,1,REPUBLICA,2019-05-14 02:27:02+00:00,SANTA ANA,2019-05-14 02:35:20+00:00,metro,-70.667568,-33.447960,-70.660112,-33.438311,88b2c5543bfffff,88b2c55411fffff
6793,m6793,35127810,LA CISTERNA,LAS CONDES,weekday,1.2806,morning,work,1,LA CISTERNA L2,2019-05-13 10:47:32+00:00,ESCUELA MILITAR,2019-05-13 11:32:15+00:00,metro,NaN,NaN,-70.584445,-33.414320,<NA>,88b2c556d3fffff


[Issue(level='info', code='SCH.DOMAIN.EMPTY_VALUES', message="El DomainSpec del campo 'time_period' tiene values vacío; se tratará como dominio bootstrap/extensible según política.", field='time_period', source_field=None, row_count=None, details={'field': 'time_period', 'values_size': 0, 'extendable': True, 'note': 'candidate_bootstrap'}),
 Issue(level='info', code='DOM.EXTENSION.APPLIED', message="Se extendió el dominio de 'time_period' con 6 valores nuevos.", field='time_period', source_field=None, row_count=None, details={'field': 'time_period', 'n_added': 6, 'added_values_sample': ['-', 'afternoon', 'evening', 'midday', 'morning'], 'added_values_total': 6, 'policy': 'extendable_non_strict', 'action': 'extended_domain'}),
 Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'destination_time_utc' falló en algunas filas (16.9%); se marcarán como nulos para validación posterior.", field='destination_time_utc', source_field=None, row_count=11